# Setup

In [ ]:
# Install (run once at the top of the notebook, in its own cell)
!pip install transformer_lens fancy_einsum einops --quiet

In [ ]:
try:
  import google.colab
  IN_COLAB = True
  print("Running as a Colab notebook")
  %pip install git+https://github.com/neelnanda-io/Easy-Transformer.git@clean-transformer-demo
  # Install another version of node that makes PySvelte work way faster
  !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash -; sudo apt-get install -y nodejs
  %pip install git+https://github.com/neelnanda-io/PySvelte.git
  %pip install fancy_einsum
  %pip install einops
except:
  IN_COLAB = False
  print("Running as a Jupyter notebook - intended for development only!")

In [ ]:
!pip install transformer_lens

In [ ]:
import einops
from fancy_einsum import einsum
from dataclasses import dataclass
from transformer_lens import HookedTransformer
import torch
import torch.nn as nn
import numpy as np
import math
from transformer_lens.utils import gelu_new, tokenize_and_concatenate, get_corner
import tqdm.auto as tqdm

In [ ]:
reference_gpt2 = HookedTransformer.from_pretrained("gpt2-small", fold_ln=False, center_unembed=False, center_writing_weights=False)

Run a reference forward pass so we have a `cache` for the tests.

In [ ]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text).cuda()
logits, cache = reference_gpt2.run_with_cache(tokens)

## Reference activation shapes

Key:
```
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 3072 (4 * d_model)
d_head = 64 (d_model / n_heads)
```

In [ ]:
for activation_name, activation in cache.cache_dict.items():
    # Only print for first layer
    if ".0." in activation_name or "blocks" not in activation_name:
        print(activation_name, activation.shape)

## Reference parameter shapes

In [ ]:
for name, param in reference_gpt2.named_parameters():
    # Only print for first layer
    if ".0." in name or "blocks" not in name:
        print(name, param.shape)

## Config

In [ ]:

@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12

cfg = Config()
print(cfg)

Key:
batch = 1
position = 35
d_model = 768
n_heads = 12
n_layers = 12
d_mlp = 4 * 768 = 3072
d_head = 768 / 12 = 64

In [ ]:
# All activation shapes of ref model
for activation_name, activation in cache.cache_dict.items():
  if ".0." in activation_name or "blocks" not in activation_name:
    print(activation_name, activation.shape)

# Actual Implementation

In [ ]:
for name, param in reference_gpt2.named_parameters():
  print(name, param.shape)


Config class

In [ ]:
@dataclass
class Config:
  d_model: int = 768
  debug: bool = True
  layer_norm_eps: float = 1e-5
  d_vocab: int = 50257
  init_range: float = 0.02
  n_ctx: int = 1024
  d_head: int = 64
  d_mlp: int = 3072
  n_heads: int = 12

cfg = Config()
print(cfg)

## Some tests

In [ ]:
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randn(shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    random_input = torch.randint(100, 1000, shape).cuda()
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    print("Output shape:", output.shape)
    print()
    return output

# takes instance of that layer from ref model and its original input
def load_gpt2_test(cls, gpt2_layer, input_name, cache_dict=cache.cache_dict):
    cfg = Config(debug=True)
    layer = cls(cfg).cuda()
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    # Allow inputs of strings or tensors
    if isinstance(input_name, str):
        reference_input = cache_dict[input_name]
    else:
        reference_input = input_name
    print("Input shape:", reference_input.shape)
    output = layer(reference_input)
    print("Output shape:", output.shape)
    reference_output = gpt2_layer(reference_input)
    print("Reference output shape:", reference_output.shape)

    comparison = torch.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum()/comparison.numel():.2%} of the values are correct")
    return output

## LayerNorm

1.  Make mean 0
2. normalize to have variance 1
3. Scale with learned weights
4. Translate with learned bias

In [ ]:
class LayerNorm(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.w = nn.Parameter(torch.ones(cfg.d_model))
    self.b = nn.Parameter(torch.zeros(cfg.d_model))

  def forward(self, residual):
    # residual: [batch, position, d_model]
    if cfg.debug: print("Residual:", residual.shape)
    residual = residual - einops.reduce(residual, "batch position d_model -> batch position 1", "mean") # making mean 0
    # Calculate variance, then sqrt it. Epsilon to prevent divide by 0
    scale = (einops.reduce(residual.pow(2), "batch position d_model -> batch position 1", "mean") + cfg.layer_norm_eps).sqrt()
    normalized = residual / scale
    normalized = normalized * self.w + self.b
    if cfg.debug: print("Normalized:", residual.shape)
    return normalized


In [ ]:
# Testing layernorm
_ = rand_float_test(LayerNorm, [2, 4, 768])

In [ ]:
_ = load_gpt2_test(LayerNorm, reference_gpt2.ln_final, "blocks.11.hook_resid_post")

## Embedding
A lookup table from tokens to residual stream vectors

In [ ]:
class Embed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_E = nn.Parameter(torch.empty((cfg.d_vocab, cfg.d_model)))
    nn.init.normal_(self.W_E, std = self.cfg.init_range)

  def forward(self, tokens):
    # tokens shape: [batch, positions]
    if cfg.debug: print("Tokens", tokens.shape)
    embed = self.W_E[tokens, :] # applying the embedding by indexing along d vocab axis, final shape = [batch, pos, d_model]
    if cfg.debug: print("Embeddings", embed.shape)
    return embed

In [ ]:
# Testing embedding layer
rand_int_test(Embed, [2, 4])
load_gpt2_test(Embed, reference_gpt2.embed, tokens)

## Positional Embedding

lookup table for positions to give each position a context instead of just each word/token like bag of words

this weight is updated on gradient descent hence the name learned absolute pos embedding

In [ ]:
class PosEmbed(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_pos = nn.Parameter(torch.empty((cfg.n_ctx, cfg.d_model)))
    nn.init.normal_(self.W_pos, std=self.cfg.init_range)

  def forward(self, tokens):
    # tokens is [batch, position]
    if cfg.debug: print("Tokens:", tokens.shape)
    pos_embed = self.W_pos[:tokens.size(1), :] # [position, d_model] -> indexing by position, so taking the size of the seq
    pos_embed = einops.repeat(pos_embed, "position d_model -> batch position d_model", batch = tokens.size(0))
    if cfg.debug: print("pos_embed:", pos_embed.shape)
    return pos_embed


In [ ]:
# Testing pos embed layer
rand_int_test(PosEmbed, [2, 4])
load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)

## Attention
1. Produce an attention pattern for each destination token - a probability dist over 0th to curr token
  * Linear map from input -> query, key, where shape: [batch, head_index, d_head]
  * then dot product every pair of queries and keys to get attention scores [batch, head_index, query_pos, key_pos] (query = dest, key = src)
  * Scale and mask attn scores to make it causal
  * softmax row-wise, to get a probability dist along each the key_pos dim -> this is the final attention pattern
2. Move info from src tokens to dest token using attention pattern (moving is via linear map)
-  Linear map from input -> value [batch, key_pos, head_index, d_head]
- Mix along the key_pos axis with attention pattern to get a mixed value [batch, query_pos, head_index, d_head]
- map to output, [batch, position, d_model] (position is query pos since we summed over all the attention heads


In [ ]:
class Attention(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.cfg = cfg
    self.W_Q = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_Q, std=self.cfg.init_range)
    self.b_Q = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
    self.W_K = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_K, std=self.cfg.init_range)
    self.b_K = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))
    self.W_V = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
    nn.init.normal_(self.W_V, std=self.cfg.init_range)
    self.b_V = nn.Parameter(torch.zeros((cfg.n_heads, cfg.d_head)))

    self.W_O = nn.Parameter(torch.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
    nn.init.normal_(self.W_O, std=self.cfg.init_range)
    self.b_O = nn.Parameter(torch.zeros((cfg.d_model)))

    self.register_buffer("IGNORE", torch.tensor(-1e5, dtype=torch.float32, device="cuda"))
  def forward(self, normalized_resid_pre):
    # normalized_resid_pre: [batch, position, d_model]
    if self.cfg.debug: print("Normalized_resid_pre:", normalized_resid_pre.shape)

    q = einsum("batch query_pos d_model, n_heads d_model d_head -> batch query_pos n_heads d_head", normalized_resid_pre, self.W_Q) + self.b_Q
    k = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_K) + self.b_K

    attn_scores = einsum("batch query_pos n_heads d_head, batch key_pos n_heads d_head -> batch n_heads query_pos key_pos", q, k)
    attn_scores = attn_scores / math.sqrt(self.cfg.d_head)
    attn_scores = self.apply_causal_mask(attn_scores)

    pattern = attn_scores.softmax(dim=-1) # [batch, n_head, query_pos, key_pos]

    v = einsum("batch key_pos d_model, n_heads d_model d_head -> batch key_pos n_heads d_head", normalized_resid_pre, self.W_V) + self.b_V

    z = einsum("batch n_heads query_pos key_pos, batch key_pos n_heads d_head -> batch query_pos n_heads d_head", pattern, v)

    attn_out = einsum("batch query_pos n_heads d_head, n_heads d_head d_model -> batch query_pos d_model", z, self.W_O) + self.b_O

    if cfg.debug:
      print("z shape:", z.shape)
      print("W_O shape:", self.W_O.shape)

    return attn_out

  # mask to remaining tokens in a sequence to maintain backward looking property properly
  def apply_causal_mask(self, attn_scores):
    mask = torch.triu(torch.ones(attn_scores.size(-2), attn_scores.size(-1), device=attn_scores.device), diagonal = 1).bool()
    attn_scores.masked_fill_(mask, self.IGNORE)
    return attn_scores



In [ ]:
# Testing attention layer
rand_float_test(Attention, [2, 4, 768])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["blocks.0.ln1.hook_normalized"])

## MLP

## Transformer Block

## Unembedding

## Full Transformer

## Tests